# NHSJS revision — Phase A & B experiments

Runs every new analysis the reviewer asked for, on the **same v2 region (Yellapur/Mundgod periphery, MGRS 43PDS)**, the **same Jan–Mar 2018 vs Jan–Mar 2024 windows**, and the **same canonical checkpoint** as the submitted paper.

| Section | Reviewer point |
|---|---|
| 1 Setup — deterministic brightness targets | R6 (run-to-run variation), reproducibility |
| 2 Earth Engine exports — QA60 vs Cloud Score+ composites, Hansen v1_11 + v1_13, WorldCover, SRTM | #6, #7 |
| 4–5 Inference + per-tile table | all |
| 6 Headline metrics with Wilson CIs, MCC, balanced accuracy, AUPRC | R7, R8 |
| 7 QA60 → Cloud Score+: how many transitions disappear | #6 |
| 8 Hansen loss-year breakdown, 2018 removed, FPs on 2024 loss | #7 |
| 9 NDWI threshold sweep + representativeness of excluded tiles | #4, #5, R2 |
| 10 Single-date sanity check vs ESA WorldCover; recall by loss fraction | #8 |
| 11 Synthetic clearing experiment | #8 (direct test of dilution) |
| 12 Hand-label + blinded audit packs, and the scoring cell | #8 |
| 13 Optional: multi-seed retraining | R6 |

**Run order:** 1 → 2 (wait for exports, ~10–30 min) → 3 onward. Everything is saved to `MyDrive/eurosat_revision/`. Paste the printed outputs back to Claude.

## 1. Setup

In [ ]:
!pip install -q geemap rasterio
import os, glob, json, math, random, time
import numpy as np, pandas as pd
import torch, torch.nn as nn
import torchvision.datasets as datasets, torchvision.transforms as transforms
from torchvision.models import resnet50
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import rasterio
import matplotlib.pyplot as plt
import ee
from google.colab import drive

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything()

drive.mount('/content/drive')
ee.Authenticate()
ee.Initialize(project='eurostatpaper')

OUT = '/content/drive/MyDrive/eurosat_revision'
os.makedirs(OUT, exist_ok=True)
MODEL_PATH = '/content/drive/MyDrive/eurosat_deforestation/best_model.pth'   # canonical checkpoint (95.68%)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dataset = datasets.EuroSAT(root='./data', download=True, transform=transforms.ToTensor())
CLASSES = dataset.classes
FOREST, SEALAKE = CLASSES.index('Forest'), CLASSES.index('SeaLake')

def load_model(path=MODEL_PATH):
    m = resnet50(weights=None)
    m.fc = nn.Linear(m.fc.in_features, len(CLASSES))
    m.load_state_dict(torch.load(path, map_location=device))
    return m.to(device).eval()
model = load_model()
print('Model loaded on', device)

### 1b. Deterministic brightness targets
The submitted pipeline drew 200 **unseeded** random EuroSAT images per session to set the moment-matching target, so the target (and therefore the predictions) changed between Colab sessions — your saved outputs show 96.79, 98.32 and ~98.2. This replaces it with the mean/std over the **whole training split** (same 70:15:15 split, seed 42), so every run is identical.

In [ ]:
labels_all = np.array(dataset.targets)
train_idx, temp_idx = train_test_split(np.arange(len(dataset)), test_size=0.30,
                                       stratify=labels_all, random_state=SEED)
loader = DataLoader(Subset(dataset, train_idx), batch_size=512, num_workers=2)
s = torch.zeros(3); s2 = torch.zeros(3); npx = 0
for x, _ in loader:
    x = x * 255
    s += x.sum(dim=(0, 2, 3)); s2 += (x ** 2).sum(dim=(0, 2, 3)); npx += x.shape[0] * 64 * 64
band_mean = (s / npx).numpy()
band_std = torch.sqrt(s2 / npx - (s / npx) ** 2).numpy()
all_mean = float((s.sum() / (3 * npx)))
all_std = float(torch.sqrt(s2.sum() / (3 * npx) - (s.sum() / (3 * npx)) ** 2))
TARGET = dict(scalar=(all_mean, all_std), per_band=(band_mean, band_std))
json.dump({'scalar': [all_mean, all_std], 'band_mean': band_mean.tolist(), 'band_std': band_std.tolist()},
          open(f'{OUT}/brightness_targets.json', 'w'), indent=1)
print(f'Scalar target  mean={all_mean:.2f} std={all_std:.2f}')
print('Per-band mean', band_mean.round(2), ' std', band_std.round(2))

## 2. Earth Engine exports
Two composites per year that differ **only** in the cloud mask (QA60 as submitted vs Cloud Score+ `cs_cdf ≥ 0.60`). The scene filter (`CLOUDY_PIXEL_PERCENTAGE < 20`), collection (`COPERNICUS/S2_HARMONIZED`, which already applies the post-2022 DN offset correction) and median compositing are unchanged, so any difference is attributable to the mask.

A reference stack goes on the identical grid: Hansen v1_11 and v1_13 `lossyear`, `treecover2000`, ESA WorldCover 2021, and SRTM elevation/slope.

In [ ]:
REGION = ee.Geometry.Rectangle([74.550, 14.750, 75.016, 15.200])   # v2 region, as submitted
WINDOWS = {'2018': ('2018-01-01', '2018-03-31'), '2024': ('2024-01-01', '2024-03-31')}
CS_THRESHOLD = 0.60
S2_BANDS = ['B4', 'B3', 'B2', 'B8']
cs_plus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')

def qa60_mask(img):
    qa = img.select('QA60')
    return img.updateMask(qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0)))

def csplus_mask(img):
    return img.updateMask(img.select('cs_cdf').gte(CS_THRESHOLD))

def s2_collection(start, end):
    return (ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
            .filterBounds(REGION).filterDate(start, end)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

def composite(start, end, method):
    col = s2_collection(start, end)
    col = col.map(qa60_mask) if method == 'qa60' else col.linkCollection(cs_plus, ['cs_cdf']).map(csplus_mask)
    col = col.select(S2_BANDS)
    med = col.median()
    nobs = col.select('B4').count().rename('nobs')
    return med.unmask(0).addBands(nobs.unmask(0)).toInt16().clip(REGION), col.size()

# Evidence for the reviewer's QA60 point: how many scenes carry ANY QA60 cloud flag over the region?
def qa60_flagged(img):
    mx = img.select('QA60').reduceRegion(ee.Reducer.max(), REGION, 600, maxPixels=1e9).get('QA60')
    return img.set('qa60_max', mx)
for yr, (a, b) in WINDOWS.items():
    col = s2_collection(a, b).map(qa60_flagged)
    vals = col.aggregate_array('qa60_max').getInfo()
    print(f'{yr}: {len(vals)} scenes, {sum(1 for v in vals if v)} with any QA60 cloud/cirrus flag')

In [ ]:
def export(img, name):
    t = ee.batch.Export.image.toDrive(image=img, description=name, folder='eurosat_revision',
                                      fileNamePrefix=name, region=REGION, scale=10, crs='EPSG:4326',
                                      maxPixels=1e10, fileDimensions=6144)   # one file per export
    t.start(); return t

tasks = {}
for method in ['qa60', 'csplus']:
    for yr, (a, b) in WINDOWS.items():
        img, n = composite(a, b, method)
        print(f'{method} {yr}: {n.getInfo()} scenes')
        tasks[f'comp_{method}_{yr}'] = export(img, f'comp_{method}_{yr}')

h111 = ee.Image('UMD/hansen/global_forest_change_2023_v1_11')
try:
    h113 = ee.Image('UMD/hansen/global_forest_change_2025_v1_13')
    print('Hansen v1_13 bands:', h113.bandNames().getInfo())
except Exception as e:
    raise SystemExit(f'v1_13 not found in Earth Engine ({e}). Tell Claude — we will use the latest available release.')

wc = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map')
dem = ee.Image('USGS/SRTMGL1_003')
ref = ee.Image.cat([
    h111.select('lossyear').rename('lossyear_v111'),
    h113.select('lossyear').rename('lossyear_v113'),
    h113.select('treecover2000'),
    wc.rename('worldcover'),
    dem.rename('elev'),
    ee.Terrain.slope(dem).rename('slope'),
]).unmask(0).toInt16().clip(REGION)
tasks['reference'] = export(ref, 'reference')
print('Started:', list(tasks))

In [ ]:
# Re-run until every task says COMPLETED (usually 10-30 min)
for k, t in tasks.items():
    st = t.status(); print(f"{k:22s} {st['state']}  {st.get('error_message', '')}")

## 3. Helper functions

In [ ]:
# === 3. Core helpers (pure numpy — tested offline before shipping) ===
import math
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

TILE = 64

def tile_reduce(arr2d, tile=TILE, how='mean'):
    """Collapse a (H, W) array to a (rows, cols) grid of per-tile statistics."""
    nr, nc = arr2d.shape[0] // tile, arr2d.shape[1] // tile
    a = arr2d[:nr * tile, :nc * tile].reshape(nr, tile, nc, tile)
    if how == 'mean':
        return a.mean(axis=(1, 3))
    if how == 'min':
        return a.min(axis=(1, 3))
    if how == 'max':
        return a.max(axis=(1, 3))
    if how == 'nanmean':
        return np.nanmean(a, axis=(1, 3))
    raise ValueError(how)

def to_tiles(img, tile=TILE):
    """(C, H, W) -> (N, C, tile, tile), row-major, same order as the original pipeline."""
    c, h, w = img.shape
    nr, nc = h // tile, w // tile
    t = img[:, :nr * tile, :nc * tile].reshape(c, nr, tile, nc, tile)
    return t.transpose(1, 3, 0, 2, 4).reshape(nr * nc, c, tile, tile), nr, nc

def _moment_match(x, mask, target_mean, target_std, low_pct=1, high_pct=99):
    vals = x[mask]
    lo, hi = np.percentile(vals, low_pct), np.percentile(vals, high_pct)
    xc = np.clip(x, lo, hi).astype(np.float32)
    m, s = xc[mask].mean(), xc[mask].std()
    y = (xc - m) / (s + 1e-6) * target_std + target_mean
    y[~mask] = 0
    return np.clip(y, 0, 255).astype(np.uint8)

def moment_match(rgb, valid, target_mean, target_std, per_band=False):
    """Same method as the submitted paper (joint 1-99% clip, scalar mean/std match),
    but statistics are computed over VALID pixels only and targets are deterministic.
    per_band=True is the ablation for reviewer #6 (matches each band separately)."""
    v = valid.astype(bool)
    if not per_band:
        return _moment_match(rgb, np.broadcast_to(v, rgb.shape),
                             float(np.mean(target_mean)), float(np.mean(target_std)))
    return np.stack([_moment_match(rgb[b], v, target_mean[b], target_std[b]) for b in range(3)])

def ndwi(b3, b8):
    b3 = b3.astype(np.float32); b8 = b8.astype(np.float32)
    with np.errstate(divide='ignore', invalid='ignore'):
        out = (b3 - b8) / (b3 + b8)
    out[(b3 + b8) == 0] = np.nan
    return out

def loss_fraction(lossyear, treecover, years, canopy=30):
    """Per-pixel 0/1 Hansen tree-cover loss in the given loss years (e.g. 18..24)
    on pixels with >= canopy % tree cover in 2000. Returns per-tile fraction."""
    hit = np.isin(lossyear, list(years)) & (treecover >= canopy)
    return tile_reduce(hit.astype(np.float32))

# ---------- metrics ----------
def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)

def score(y_true, y_pred, y_score=None):
    y_true = np.asarray(y_true, bool); y_pred = np.asarray(y_pred, bool)
    tp = int((y_true & y_pred).sum()); fp = int((~y_true & y_pred).sum())
    fn = int((y_true & ~y_pred).sum()); tn = int((~y_true & ~y_pred).sum())
    n = tp + fp + fn + tn
    prec = tp / (tp + fp) if tp + fp else np.nan
    rec = tp / (tp + fn) if tp + fn else np.nan
    spec = tn / (tn + fp) if tn + fp else np.nan
    denom = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = (tp * tn - fp * fn) / denom if denom else np.nan
    out = dict(n=n, TP=tp, FP=fp, FN=fn, TN=tn,
               base_rate=(tp + fn) / n if n else np.nan,
               precision=prec, prec_lo=wilson(tp, tp + fp)[0], prec_hi=wilson(tp, tp + fp)[1],
               recall=rec, rec_lo=wilson(tp, tp + fn)[0], rec_hi=wilson(tp, tp + fn)[1],
               specificity=spec, balanced_acc=(rec + spec) / 2, mcc=mcc,
               agreement=(tp + tn) / n if n else np.nan)
    if y_score is not None and y_true.any():
        out['auprc'] = average_precision_score(y_true, y_score)
    return out

def fmt(s):
    """One-line summary for the paper/table."""
    return (f"n={s['n']}  TP={s['TP']} FP={s['FP']} FN={s['FN']} TN={s['TN']}  "
            f"P={s['precision']:.3f} [{s['prec_lo']:.3f}-{s['prec_hi']:.3f}]  "
            f"R={s['recall']:.3f} [{s['rec_lo']:.3f}-{s['rec_hi']:.3f}]  "
            f"base={s['base_rate']:.3f}  MCC={s['mcc']:.3f}  BalAcc={s['balanced_acc']:.3f}"
            + (f"  AUPRC={s['auprc']:.3f}" if 'auprc' in s else ''))

# ---------- synthetic clearing ----------
def insert_patch(forest_tile, donor_tile, frac, rng, mode='block', block=4):
    """Paste `frac` of donor_tile's area into forest_tile.
    mode='block': one contiguous square (a single clearing).
    mode='scattered': many block x block squares at random positions (diffuse loss)."""
    t = forest_tile.copy()
    _, h, w = t.shape
    if frac <= 0:
        return t, 0.0
    if mode == 'block':
        side = int(round(h * math.sqrt(frac)))
        side = max(1, min(side, h))
        y0 = rng.integers(0, h - side + 1); x0 = rng.integers(0, w - side + 1)
        t[:, y0:y0 + side, x0:x0 + side] = donor_tile[:, y0:y0 + side, x0:x0 + side]
        return t, side * side / (h * w)
    cells = [(y, x) for y in range(0, h, block) for x in range(0, w, block)]
    k = max(1, int(round(frac * len(cells))))
    for i in rng.choice(len(cells), size=min(k, len(cells)), replace=False):
        y, x = cells[i]
        t[:, y:y + block, x:x + block] = donor_tile[:, y:y + block, x:x + block]
    return t, min(k, len(cells)) * block * block / (h * w)

## 4. Load rasters and run inference

In [ ]:
def load_tif(name):
    files = sorted(glob.glob(f'{OUT}/{name}*.tif'))
    assert len(files) == 1, f'{name}: expected 1 file, found {files}'
    with rasterio.open(files[0]) as src:
        return src.read(), src.transform

ref_arr, transform = load_tif('reference')
LOSS111, LOSS113, TC2000, WC, ELEV, SLOPE = ref_arr
comps = {}
for method in ['qa60', 'csplus']:
    for yr in WINDOWS:
        arr, tr = load_tif(f'comp_{method}_{yr}')
        assert arr.shape[1:] == ref_arr.shape[1:] and tr == transform, 'grids differ — tell Claude'
        comps[(method, yr)] = arr          # bands: B4 B3 B2 B8 nobs
print('Grid', ref_arr.shape[1:], '| pixel size (deg)', transform.a, -transform.e)

# True tile size (paper says ~41 ha; check it)
lat0 = transform.f + transform.e * ref_arr.shape[1] / 2
px_ns = -transform.e * 111_320; px_ew = transform.a * 111_320 * math.cos(math.radians(lat0))
print(f'Pixel ~{px_ew:.2f} m E-W x {px_ns:.2f} m N-S -> tile ~{64*px_ew:.0f} x {64*px_ns:.0f} m = {64*px_ew*64*px_ns/1e4:.1f} ha')

In [ ]:
@torch.no_grad()
def predict(tiles, m=None, bs=512):
    m = model if m is None else m
    out = []
    for i in range(0, len(tiles), bs):
        x = torch.from_numpy(tiles[i:i + bs]).float().div(255).to(device)
        out.append(torch.softmax(m(x), 1).cpu().numpy())
    return np.concatenate(out)

def run_year(method, yr, per_band=False, m=None):
    arr = comps[(method, yr)]
    rgb, b3, b8, nobs = arr[[0, 1, 2]], arr[1], arr[3], arr[4]
    valid = nobs > 0
    tm, ts = TARGET['per_band'] if per_band else TARGET['scalar']
    matched = moment_match(rgb.astype(np.float32), valid, tm, ts, per_band=per_band)
    tiles, nr, nc = to_tiles(matched)
    probs = predict(tiles, m)
    nd = ndwi(b3, b8); nd[~valid] = np.nan
    return dict(probs=probs, tiles=tiles, nr=nr, nc=nc,
                valid=tile_reduce(valid.astype(np.float32)).ravel(),
                ndwi=tile_reduce(nd, how='nanmean').ravel())

RUNS = {}
for method in ['qa60', 'csplus']:
    for yr in WINDOWS:
        RUNS[(method, yr)] = run_year(method, yr)
        print(method, yr, 'done')
NR, NC = RUNS[('csplus', '2018')]['nr'], RUNS[('csplus', '2018')]['nc']
print('Tiles:', NR * NC)

## 5. Per-tile table

In [ ]:
rows, cols = np.divmod(np.arange(NR * NC), NC)
base = pd.DataFrame({'row': rows, 'col': cols})
# tile centre coordinates (for hand-labelling in Google Earth Pro)
base['lon'] = transform.c + (cols * 64 + 32) * transform.a
base['lat'] = transform.f + (rows * 64 + 32) * transform.e
for y in range(18, 26):
    base[f'loss113_{2000+y}'] = loss_fraction(LOSS113, TC2000, [y]).ravel()
for y in range(18, 24):
    base[f'loss111_{2000+y}'] = loss_fraction(LOSS111, TC2000, [y]).ravel()
base['treecover2000'] = tile_reduce(TC2000.astype(np.float32)).ravel()
base['wc_tree'] = tile_reduce((WC == 10).astype(np.float32)).ravel()
base['wc_crop'] = tile_reduce((WC == 40).astype(np.float32)).ravel()
base['wc_built'] = tile_reduce((WC == 50).astype(np.float32)).ravel()
base['wc_water'] = tile_reduce((WC == 80).astype(np.float32)).ravel()
base['elev'] = tile_reduce(ELEV.astype(np.float32)).ravel()
base['slope'] = tile_reduce(SLOPE.astype(np.float32)).ravel()

def loss_any(df, version, years):
    cols_ = [f'loss{version}_{y}' for y in years]
    return df[cols_].sum(axis=1)

def build(method, ndwi_thr=0.0, min_valid=0.99, runs=None):
    runs = runs or RUNS
    df = base.copy()
    for yr in WINDOWS:
        r = runs[(method, yr)]
        p = r['probs']
        df[f'cls_{yr}'] = np.array(CLASSES)[p.argmax(1)]
        df[f'conf_{yr}'] = p.max(1)
        df[f'pforest_{yr}'] = p[:, FOREST]
        df[f'ndwi_{yr}'] = r['ndwi']
        df[f'valid_{yr}'] = r['valid']
        df[f'ambig_{yr}'] = (df[f'cls_{yr}'] == 'SeaLake') & ~(df[f'ndwi_{yr}'] > ndwi_thr)
    df['ambiguous'] = df['ambig_2018'] | df['ambig_2024']
    df['cloudgap'] = (df['valid_2018'] < min_valid) | (df['valid_2024'] < min_valid)
    df['evaluable'] = ~df['ambiguous'] & ~df['cloudgap']
    df['model_pos'] = (df['cls_2018'] == 'Forest') & (df['cls_2024'] != 'Forest')
    df['model_score'] = df['pforest_2018'] * (1 - df['pforest_2024'])
    return df

DF = {m: build(m) for m in ['qa60', 'csplus']}
for m, df in DF.items():
    df.to_csv(f'{OUT}/tiles_{m}.csv', index=False)
    print(m, '| evaluable', df.evaluable.sum(), '| NDWI-ambiguous', df.ambiguous.sum(), '| cloud-gap', df.cloudgap.sum())

## 6. Headline metrics (Wilson 95% CIs, MCC, balanced accuracy, AUPRC)

In [ ]:
REFS = {
    'v1_11 2018-2023 (as submitted)': ('111', range(2018, 2024)),
    'v1_13 2018-2024':                ('113', range(2018, 2025)),
    'v1_13 2019-2024 (2018 dropped)': ('113', range(2019, 2025)),
    'v1_13 2019-2023':                ('113', range(2019, 2024)),
}
results = []
for m, df in DF.items():
    d = df[df.evaluable]
    for name, (v, yrs) in REFS.items():
        s = score(loss_any(d, v, yrs) > 0, d.model_pos, d.model_score)
        s.update(composite=m, reference=name); results.append(s)
        print(f'{m:7s} | {name:32s} | {fmt(s)}')
pd.DataFrame(results).to_csv(f'{OUT}/headline_metrics.csv', index=False)

## 7. QA60 → Cloud Score+: raw change rate and disappearing transitions (reviewer #6)

In [ ]:
q, c = DF['qa60'], DF['csplus']
for m, df in DF.items():
    raw = (df.cls_2018 != df.cls_2024).mean()
    print(f"{m:7s} raw change rate {raw:.1%} | Forest tiles 2018={(df.cls_2018=='Forest').sum()} "
          f"2024={(df.cls_2024=='Forest').sum()} | Forest->other {df.model_pos.sum()} | "
          f"Herb->Forest {((df.cls_2018=='HerbaceousVegetation')&(df.cls_2024=='Forest')).sum()}")
both = q.evaluable & c.evaluable
gone = (q.model_pos & ~c.model_pos & both).sum()
new = (~q.model_pos & c.model_pos & both).sum()
print(f'\nForest->other transitions under QA60 that DISAPPEAR with Cloud Score+: {gone} of {(q.model_pos & both).sum()}')
print(f'New transitions that appear only with Cloud Score+: {new}')
print(f'Any-class transitions that disappear: {((q.cls_2018!=q.cls_2024)&(c.cls_2018==c.cls_2024)).sum()}')
print(f"2018 labels that change between masks: {(q.cls_2018!=c.cls_2018).mean():.1%} | 2024: {(q.cls_2024!=c.cls_2024).mean():.1%}")

# Normalisation ablation: per-band moment matching on the Cloud Score+ composites
RUNS_PB = {('csplus', yr): run_year('csplus', yr, per_band=True) for yr in WINDOWS}
pb = build('csplus', runs=RUNS_PB)
print(f"\nPer-band normalisation (Cloud Score+): raw change {(pb.cls_2018!=pb.cls_2024).mean():.1%}, "
      f"Forest 2018={(pb.cls_2018=='Forest').sum()} 2024={(pb.cls_2024=='Forest').sum()}, Forest->other {pb.model_pos.sum()}")
d = pb[pb.evaluable]
print('Per-band vs v1_13 2019-2024:', fmt(score(loss_any(d,'113',range(2019,2025))>0, d.model_pos, d.model_score)))

## 8. Hansen loss-year breakdown (reviewer #7)

In [ ]:
df = DF['csplus']; d = df[df.evaluable].copy()
ycols = [f'loss113_{y}' for y in range(2018, 2026)]
d['gfw_pos'] = loss_any(d, '113', range(2018, 2025)) > 0
d['cat'] = np.select([d.gfw_pos & d.model_pos, ~d.gfw_pos & d.model_pos, d.gfw_pos & ~d.model_pos],
                     ['TP', 'FP', 'FN'], 'TN')
has = d[ycols].gt(0)
d['dominant_year'] = np.where(has.any(axis=1), d[ycols].idxmax(axis=1).str[-4:], 'none')
print('Tiles with any loss in each year, by category (v1_13, 2018-2024 reference):')
print(has.groupby(d['cat']).sum().rename(columns=lambda s: s[-4:]))
print('\nDominant loss year by category:')
print(pd.crosstab(d['cat'], d['dominant_year']))

# The reviewer's exact question: of the false positives against the SUBMITTED reference
# (v1_11, any loss 2018-2023), how many sit on 2024 loss in v1_13?
fp_sub = d[d.model_pos & ~(loss_any(d, '111', range(2018, 2024)) > 0)]
print(f'\nFPs vs submitted reference: {len(fp_sub)}; of these on v1_13 2024 loss: {(fp_sub.loss113_2024 > 0).sum()}'
      f' (2025 loss, outside window: {(fp_sub.loss113_2025 > 0).sum()})')
# Tiles where the ONLY loss is 2018 (could pre-date the Jan-Mar 2018 composite)
only18 = d.loss113_2018.gt(0) & ~(loss_any(d, '113', range(2019, 2025)) > 0)
print(f'Tiles whose only loss is 2018: {only18.sum()} (FN among them: {(only18 & ~d.model_pos).sum()})')
d.to_csv(f'{OUT}/tiles_csplus_categorised.csv', index=False)

## 9. NDWI threshold sweep and representativeness of excluded tiles (reviewer #4, #5)

In [ ]:
sweep = []
for thr in [-0.3, -0.2, -0.1, -0.05, 0.0, 0.05, 0.1, 0.2]:
    df_t = build('csplus', ndwi_thr=thr)
    gfw = loss_any(df_t, '113', range(2019, 2025)) > 0
    d = df_t[df_t.evaluable]
    s = score(loss_any(d, '113', range(2019, 2025)) > 0, d.model_pos)
    sweep.append(dict(thr=thr, sealake_preds=int((df_t.cls_2018=='SeaLake').sum()+(df_t.cls_2024=='SeaLake').sum()),
                      excluded=int(df_t.ambiguous.sum()), excluded_pct=df_t.ambiguous.mean(),
                      gfw_pos_excluded_pct=(df_t.ambiguous & gfw).sum() / gfw.sum(),
                      precision=s['precision'], recall=s['recall'], mcc=s['mcc']))
print(pd.DataFrame(sweep).round(3).to_string(index=False))

df = DF['csplus']
print('\nWorldCover water fraction in NDWI-ambiguous tiles (is any of it really water?):',
      df.loc[df.ambiguous, 'wc_water'].describe().round(3).to_dict())

cmp = df.groupby('ambiguous')[['elev', 'slope', 'treecover2000', 'wc_tree', 'wc_crop', 'wc_built']].mean().round(2)
cmp['gfw_pos_rate'] = df.groupby('ambiguous').apply(lambda g: (loss_any(g, '113', range(2019, 2025)) > 0).mean()).round(3)
cmp = cmp.rename(index={False: 'kept', True: 'excluded (NDWI-ambiguous)'})
print('\nKept vs excluded tiles:'); print(cmp.to_string())

## 10. Single-date sanity check against ESA WorldCover 2021 (reviewer #8, before hand-labelling)
WorldCover is a 2021 map and itself a model, so this is a **proxy**, not ground truth — but it is independent of Hansen and tells us quickly whether the classifier recognises Western Ghats forest at all. Hand labels in §12 are the real test.

In [ ]:
df = DF['csplus']; d = df[df.evaluable]
wc_forest = d.wc_tree >= 0.5
for yr in WINDOWS:
    s = score(wc_forest, d[f'cls_{yr}'] == 'Forest')
    print(f"{yr}: WorldCover-forest tiles {wc_forest.sum()} | model calls Forest {(d[f'cls_{yr}']=='Forest').sum()} | "
          f"forest recall {s['recall']:.3f} [{s['rec_lo']:.3f}-{s['rec_hi']:.3f}] | forest precision {s['precision']:.3f} | "
          f"agreement {s['agreement']:.3f}")
print('\nWhat the model calls WorldCover tree-cover tiles (2018):')
print(d.loc[wc_forest, 'cls_2018'].value_counts().to_string())

# Recall by GFW loss fraction, restricted to tiles WorldCover says were forest
d = d.copy(); d['lossfrac'] = loss_any(d, '113', range(2019, 2025))
bins = [0, 0.01, 0.02, 0.05, 0.10, 1.01]
g = d[(d.lossfrac > 0) & wc_forest.loc[d.index]].copy()
g['bin'] = pd.cut(g.lossfrac, bins, right=False)
tab = g.groupby('bin', observed=False).agg(n=('model_pos', 'size'), detected=('model_pos', 'sum'))
tab['recall'] = tab.detected / tab.n
tab['ci'] = [wilson(k, n) if n else (np.nan, np.nan) for k, n in zip(tab.detected, tab.n)]
print('\nRecall by loss fraction (WorldCover-forest tiles only):'); print(tab.to_string())
print(f'2018 forest-classified share among these: {(g.cls_2018=="Forest").mean():.1%}')

## 11. Synthetic clearing experiment (direct test of sub-tile dilution)
Take tiles the model confidently calls Forest in **both** years that have **no** Hansen loss and ≥90% WorldCover tree cover. Paste real cleared-land pixels (from confident non-forest tiles with <10% tree cover) into them at increasing area fractions — one contiguous clearing (`block`) or many 40 m patches (`scattered`, closer to the diffuse loss you measured). If dilution is the mechanism, the tile should only flip once the clearing is large.

In [ ]:
df = DF['csplus']; r24 = RUNS[('csplus', '2024')]
rng = np.random.default_rng(SEED)
fpool = df.index[(df.cls_2018 == 'Forest') & (df.cls_2024 == 'Forest') & (df.conf_2018 >= 0.9) & (df.conf_2024 >= 0.9)
                 & (loss_any(df, '113', range(2018, 2026)) == 0) & (df.wc_tree >= 0.9) & (df.valid_2024 >= 0.99)].to_numpy()
DONOR_CLASSES = ['HerbaceousVegetation', 'AnnualCrop', 'Pasture', 'Residential']
dpool = df.index[df.cls_2024.isin(DONOR_CLASSES) & (df.conf_2024 >= 0.9) & (df.wc_tree < 0.1) & (df.valid_2024 >= 0.99)].to_numpy()
print(f'Forest pool {len(fpool)} | donor pool {len(dpool)} ({df.loc[dpool, "cls_2024"].value_counts().to_dict()})')
assert len(fpool) >= 20 and len(dpool) >= 20, 'Pools too small — paste this output to Claude to relax the filters'

N_TILES = min(200, len(fpool))
FRACS = [0, 0.005, 0.01, 0.025, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.75, 1.0]
sel = rng.choice(fpool, N_TILES, replace=False)
recs, batch, meta = [], [], []
for mode in ['block', 'scattered']:
    for i in sel:
        donor = rng.choice(dpool)
        for f in FRACS:
            t, real = insert_patch(r24['tiles'][i], r24['tiles'][donor], f, rng, mode=mode)
            batch.append(t); meta.append((mode, int(i), int(donor), f, real))
P = predict(np.stack(batch))
syn = pd.DataFrame(meta, columns=['mode', 'tile', 'donor', 'frac', 'real_frac'])
syn['p_forest'] = P[:, FOREST]; syn['pred'] = np.array(CLASSES)[P.argmax(1)]; syn['flipped'] = syn.pred != 'Forest'
syn.to_csv(f'{OUT}/synthetic_clearing.csv', index=False)

summ = syn.groupby(['mode', 'frac']).agg(flip_rate=('flipped', 'mean'), n=('flipped', 'size'),
                                         median_pforest=('p_forest', 'median')).reset_index()
summ['lo'], summ['hi'] = zip(*[wilson(round(r*n), n) for r, n in zip(summ.flip_rate, summ.n)])
print(summ.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4.5))
for mode, g in summ.groupby('mode'):
    ax.plot(g.frac * 100, g.flip_rate, marker='o', label=mode)
    ax.fill_between(g.frac * 100, g.lo, g.hi, alpha=0.2)
real = loss_any(df, '113', range(2019, 2025)); real = real[real > 0]
ax.axvline(real.median() * 100, ls='--', c='grey'); ax.text(real.median()*100 + 1, 0.9, f'median real loss\n{real.median():.1%}', fontsize=8)
ax.set_xlabel('Cleared area inserted (% of tile)'); ax.set_ylabel('Share of tiles no longer predicted Forest')
ax.set_ylim(-0.02, 1.02); ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f'{OUT}/fig_synthetic_clearing.png', dpi=300); plt.show()
print(f'Real GFW-positive tiles with loss >= 10%: {(real >= 0.10).sum()} of {len(real)}')

## 12. Hand-labelling and blinded audit packs
**Pack A — random single-date sample (150 tiles).** No model output, no Hansen overlay. Label each tile's dominant land cover in 2018 and 2024 using the chip plus Google Earth Pro historical imagery at the given lat/lon. Use the 10 EuroSAT class names, or `Unclear`.

**Pack B — blinded audit (stratified TP/FP/FN/TN).** Chips show both dates, model class + score, and Hansen loss pixels (orange = 2018, red = 2019–2024). The category is hidden; `audit_key.csv` holds it. Don't open the key until labelling is done.

In [ ]:
from PIL import Image, ImageDraw, ImageFont
try:
    FONT = ImageFont.load_default(size=26)
except TypeError:
    FONT = ImageFont.load_default()
df = DF['csplus']; d = pd.read_csv(f'{OUT}/tiles_csplus_categorised.csv')
r18, r24 = RUNS[('csplus', '2018')], RUNS[('csplus', '2024')]
rgb18 = moment_match(comps[('csplus','2018')][[0,1,2]].astype(np.float32), comps[('csplus','2018')][4] > 0, *TARGET['scalar'])
rgb24 = moment_match(comps[('csplus','2024')][[0,1,2]].astype(np.float32), comps[('csplus','2024')][4] > 0, *TARGET['scalar'])

def context(rgb, r, c, pad=64):
    H, W = rgb.shape[1:]
    y0, x0 = max(r*64 - pad, 0), max(c*64 - pad, 0)
    y1, x1 = min(r*64 + 64 + pad, H), min(c*64 + 64 + pad, W)
    return rgb[:, y0:y1, x0:x1].transpose(1, 2, 0), (r*64 - y0, c*64 - x0)

def chip(r, c, overlay=False, scale=4):
    panels = []
    for rgb in (rgb18, rgb24):
        img, (oy, ox) = context(rgb, r, c)
        img = img.copy()
        if overlay:
            ly = LOSS113[max(r*64-64,0):max(r*64-64,0)+img.shape[0], max(c*64-64,0):max(c*64-64,0)+img.shape[1]]
            tc = TC2000[max(r*64-64,0):max(r*64-64,0)+img.shape[0], max(c*64-64,0):max(c*64-64,0)+img.shape[1]]
            img[(ly == 18) & (tc >= 30)] = [255, 160, 0]
            img[(ly >= 19) & (ly <= 24) & (tc >= 30)] = [255, 0, 0]
        im = Image.fromarray(img).resize((img.shape[1]*scale, img.shape[0]*scale), Image.NEAREST)
        ImageDraw.Draw(im).rectangle([ox*scale, oy*scale, (ox+64)*scale-1, (oy+64)*scale-1], outline=(0, 255, 255), width=3)
        panels.append(im)
    W = sum(p.width for p in panels) + 20; H = max(p.height for p in panels)
    out = Image.new('RGB', (W, H), 'white'); out.paste(panels[0], (0, 0)); out.paste(panels[1], (panels[0].width + 20, 0))
    return out

def gmaps(lat, lon): return f'https://www.google.com/maps/@{lat:.5f},{lon:.5f},700m/data=!3m1!1e3'

rng = np.random.default_rng(SEED)
# Pack A
os.makedirs(f'{OUT}/packA', exist_ok=True)
A = d.sample(150, random_state=SEED).reset_index(drop=True)
A['label_id'] = [f'A{i:03d}' for i in rng.permutation(len(A))]
for _, t in A.iterrows():
    chip(t.row, t.col).save(f'{OUT}/packA/{t.label_id}.png')
A['maps_link'] = [gmaps(a, b) for a, b in zip(A.lat, A.lon)]
sheet = A[['label_id', 'lat', 'lon', 'maps_link']].sort_values('label_id')
for col in ['label_2018', 'label_2024', 'forest_2018_YN', 'forest_2024_YN', 'confidence_1to3', 'notes']: sheet[col] = ''
sheet.to_csv(f'{OUT}/packA_labels.csv', index=False)
A[['label_id', 'row', 'col']].to_csv(f'{OUT}/packA_key.csv', index=False)

# Pack B
os.makedirs(f'{OUT}/packB', exist_ok=True)
parts = []
for cat, n in [('TP', 15), ('FP', 20), ('FN', 20), ('TN', 10)]:
    pool = d[d.cat == cat]
    parts.append(pool.sample(min(n, len(pool)), random_state=SEED))
B = pd.concat(parts).reset_index(drop=True)
B['audit_id'] = [f'B{i:03d}' for i in rng.permutation(len(B))]
for _, t in B.iterrows():
    im = chip(t.row, t.col, overlay=True)
    draw = ImageDraw.Draw(im)
    draw.text((6, 6), f'2018: {t.cls_2018} ({t.conf_2018:.2f})', fill='yellow', font=FONT, stroke_width=2, stroke_fill='black')
    draw.text((im.width//2 + 16, 6), f'2024: {t.cls_2024} ({t.conf_2024:.2f})', fill='yellow', font=FONT, stroke_width=2, stroke_fill='black')
    im.save(f'{OUT}/packB/{t.audit_id}.png')
B['maps_link'] = [gmaps(a, b) for a, b in zip(B.lat, B.lon)]
bs = B[['audit_id', 'lat', 'lon', 'maps_link']].sort_values('audit_id')
for col in ['true_2018', 'true_2024', 'real_clearing_visible_YN', 'clearing_pct_est', 'model_2018_correct_YN',
            'model_2024_correct_YN', 'likely_cause', 'notes']: bs[col] = ''
bs.to_csv(f'{OUT}/packB_audit.csv', index=False)
B[['audit_id', 'cat', 'row', 'col']].to_csv(f'{OUT}/audit_key.csv', index=False)
print(f'Pack A: {len(A)} chips | Pack B: {len(B)} chips ({B.cat.value_counts().to_dict()})')

### 12b. Score the hand labels (run after filling `packA_labels.csv` and `packB_audit.csv`)
For `likely_cause` in Pack B use one of: `dilution` (real clearing, small share of tile), `misclass_2018`, `misclass_2024`, `misclass_both`, `real_change_missed_by_hansen`, `hansen_not_visible`, `other`.

In [ ]:
A = pd.read_csv(f'{OUT}/packA_labels.csv').merge(pd.read_csv(f'{OUT}/packA_key.csv'), on='label_id')
A = A.merge(DF['csplus'][['row', 'col', 'cls_2018', 'cls_2024']], on=['row', 'col'])
for yr in WINDOWS:
    ok = A[f'label_{yr}'].notna() & (A[f'label_{yr}'] != 'Unclear')
    a = A[ok]
    k = (a[f'label_{yr}'] == a[f'cls_{yr}']).sum()
    print(f'{yr}: 10-class accuracy {k}/{len(a)} = {k/len(a):.3f} {tuple(round(float(x),3) for x in wilson(k, len(a)))}')
    s = score(a[f'label_{yr}'] == 'Forest', a[f'cls_{yr}'] == 'Forest')
    print(f"      forest vs non-forest: agreement {s['agreement']:.3f}, forest recall {s['recall']:.3f} "
          f"[{s['rec_lo']:.3f}-{s['rec_hi']:.3f}], forest precision {s['precision']:.3f}")
    print(pd.crosstab(a[f'label_{yr}'], a[f'cls_{yr}'], rownames=['true'], colnames=['pred']))

B = pd.read_csv(f'{OUT}/packB_audit.csv').merge(pd.read_csv(f'{OUT}/audit_key.csv'), on='audit_id')
print('\nAudit — likely cause by category:'); print(pd.crosstab(B.cat, B.likely_cause))
print('\nReal clearing visible by category:'); print(pd.crosstab(B.cat, B.real_clearing_visible_YN))

## 13. Optional — multi-seed retraining (reviewer R6)
Retrains the classifier with 5 seeds (same recipe as the paper, ~7 min each on a T4) and reruns the Cloud Score+ change detection for each. cuDNN determinism is on, so each seed is itself repeatable.

In [ ]:
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, stratify=labels_all[temp_idx], random_state=SEED)

def train_seed(seed, epochs=15):
    seed_everything(seed)
    from torchvision.models import ResNet50_Weights
    m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    for p in m.parameters(): p.requires_grad = False
    for p in m.layer4.parameters(): p.requires_grad = True
    m.fc = nn.Linear(m.fc.in_features, len(CLASSES)); m = m.to(device)
    g = torch.Generator().manual_seed(seed)
    tl = DataLoader(Subset(dataset, train_idx), batch_size=32, shuffle=True, num_workers=2, generator=g)
    vl = DataLoader(Subset(dataset, val_idx), batch_size=256, num_workers=2)
    te = DataLoader(Subset(dataset, test_idx), batch_size=256, num_workers=2)
    opt = torch.optim.Adam(filter(lambda p: p.requires_grad, m.parameters()), lr=1e-4)
    crit = nn.CrossEntropyLoss(); best = (1e9, None)
    def run(loader, train):
        m.train(train); tot = cor = 0; ls = 0.0
        with torch.set_grad_enabled(train):
            for x, y in loader:
                x, y = x.to(device), y.to(device); o = m(x); l = crit(o, y)
                if train: opt.zero_grad(); l.backward(); opt.step()
                ls += l.item() * len(y); cor += (o.argmax(1) == y).sum().item(); tot += len(y)
        return ls / tot, cor / tot
    for ep in range(epochs):
        run(tl, True); vloss, vacc = run(vl, False)
        if vloss < best[0]: best = (vloss, {k: v.detach().clone() for k, v in m.state_dict().items()}, vacc, ep + 1)
    m.load_state_dict(best[1]); m.eval()
    _, tacc = run(te, False)
    path = f'{OUT}/model_seed{seed}.pth'; torch.save(m.state_dict(), path)
    return m, dict(seed=seed, best_epoch=best[3], val_acc=best[2], test_acc=tacc)

seed_rows = []
for sd in [0, 1, 2, 3, 4]:
    m, info = train_seed(sd)
    runs = {('csplus', yr): run_year('csplus', yr, m=m) for yr in WINDOWS}
    dfm = build('csplus', runs=runs); dd = dfm[dfm.evaluable]
    s = score(loss_any(dd, '113', range(2019, 2025)) > 0, dd.model_pos, dd.model_score)
    info.update(forest_2018=int((dfm.cls_2018=='Forest').sum()), model_pos=int(dfm.model_pos.sum()),
                precision=s['precision'], recall=s['recall'], mcc=s['mcc'])
    seed_rows.append(info); print(info)
seeds = pd.DataFrame(seed_rows); seeds.to_csv(f'{OUT}/seed_variation.csv', index=False)
print(seeds.describe().loc[['min', 'max', 'mean', 'std']].round(4).to_string())